# Chapter 14 - Hierarchical Clustering and DBSCAN

K-Means is fast and intuitive, but it assumes spherical clusters and requires you
to specify K up front. This notebook introduces two alternatives that relax those
assumptions:

- **Agglomerative hierarchical clustering** — builds a tree (dendrogram) of merges,
  letting you choose the number of clusters *after* fitting.
- **DBSCAN** — a density-based method that discovers clusters of arbitrary shape and
  automatically identifies outliers.

**Topics covered:**
- Agglomerative clustering: bottom-up merging
- Linkage methods: ward, complete, average, single
- Dendrograms with scipy
- AgglomerativeClustering in sklearn
- Cutting the dendrogram
- DBSCAN: core, border, and noise points
- eps and min_samples parameters
- DBSCAN advantages
- Comparing K-Means vs Hierarchical vs DBSCAN on different data shapes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN,
)
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline

---
## 1. Agglomerative Hierarchical Clustering

Agglomerative clustering is a **bottom-up** approach:

1. Start with each point as its own cluster (N clusters).
2. Find the two closest clusters and **merge** them.
3. Repeat until everything is one big cluster (or until you reach a desired number).

The result is a binary tree structure called a **dendrogram** that records the entire
merge history. You can "cut" the tree at any level to obtain a specific number of
clusters — no need to rerun the algorithm.

In [ ]:
# Small dataset to illustrate the merge process
X_small = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8],
                     [1, 0.6], [9, 11], [8, 2], [10, 2], [9, 3]])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_small[:, 0], X_small[:, 1], s=100, edgecolors='k', c='steelblue')
for i, (x, y) in enumerate(X_small):
    ax.annotate(f'  P{i}', (x, y), fontsize=11)
ax.set_title('9 sample points for hierarchical clustering')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()

---
## 2. Linkage Methods

The key question is: how do we measure the distance between two *clusters* (not just
two points)? The answer is the **linkage** criterion:

| Linkage | Distance between clusters A and B |
|---------|-----------------------------------|
| **single** | Minimum distance between any pair of points (one from A, one from B) |
| **complete** | Maximum distance between any pair |
| **average** | Average distance between all pairs |
| **ward** | Increase in total within-cluster variance after merging (minimises WCSS, like K-Means) |

**Ward** linkage tends to produce compact, spherical clusters and is the default in
sklearn's `AgglomerativeClustering`. **Single** linkage can find elongated, chain-like
clusters but is sensitive to noise.

In [ ]:
# Compute linkage matrices using scipy
methods = ['single', 'complete', 'average', 'ward']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, method in zip(axes.ravel(), methods):
    Z = linkage(X_small, method=method)
    dendrogram(Z, ax=ax, labels=[f'P{i}' for i in range(len(X_small))])
    ax.set_title(f'Linkage: {method}')
    ax.set_ylabel('Distance')

plt.suptitle('Dendrograms with different linkage methods', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Reading a Dendrogram

A dendrogram encodes the full merge history:

- **Leaves** (bottom) represent individual data points.
- Each horizontal line connecting two branches represents a **merge**.
- The **height** of the merge line indicates the distance between the two clusters
  being merged.
- A horizontal cut at any height produces a specific number of clusters: count the
  number of vertical lines the cut crosses.

Large vertical jumps in the dendrogram suggest a natural number of clusters — the
algorithm had to "reach far" to merge the next pair.

In [ ]:
# Larger dataset with clear clusters
X_hier, y_hier_true = make_blobs(n_samples=150, centers=4, cluster_std=0.6,
                                  random_state=42)

Z_ward = linkage(X_hier, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z_ward, ax=ax, truncate_mode='lastp', p=20,
           show_leaf_counts=True, leaf_rotation=90)
ax.axhline(y=8, color='red', linestyle='--', linewidth=2,
           label='Cut at distance 8 -> 4 clusters')
ax.set_title('Ward linkage dendrogram (truncated)')
ax.set_xlabel('Cluster size')
ax.set_ylabel('Distance')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Cutting the Dendrogram

We can use `scipy.cluster.hierarchy.fcluster` to cut the linkage tree at a chosen
distance threshold or to a desired number of clusters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, n_clust in zip(axes, [2, 4, 6]):
    labels = fcluster(Z_ward, t=n_clust, criterion='maxclust')
    ax.scatter(X_hier[:, 0], X_hier[:, 1], c=labels, cmap='Set1',
               edgecolors='k', s=40)
    ax.set_title(f'Cut to {n_clust} clusters')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Cutting the dendrogram at different levels', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. AgglomerativeClustering in Scikit-Learn

Scikit-learn provides `AgglomerativeClustering` which integrates cleanly with the
rest of the sklearn API. You specify `n_clusters` and optionally the `linkage` method.

In [ ]:
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg_labels = agg.fit_predict(X_hier)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_hier[:, 0], X_hier[:, 1], c=agg_labels, cmap='Set2',
           edgecolors='k', s=50)
ax.set_title('AgglomerativeClustering (ward, n_clusters=4)')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()

print(f"Cluster labels: {np.unique(agg_labels)}")
print(f"Points per cluster: {np.bincount(agg_labels)}")

---
## 6. Comparing Linkage Methods Visually

Different linkage methods produce different cluster shapes. Let us compare them on
the same data.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, link_method in zip(axes, ['ward', 'complete', 'average', 'single']):
    agg_m = AgglomerativeClustering(n_clusters=4, linkage=link_method)
    labels_m = agg_m.fit_predict(X_hier)
    ax.scatter(X_hier[:, 0], X_hier[:, 1], c=labels_m, cmap='Set2',
               edgecolors='k', s=40)
    ax.set_title(f'Linkage: {link_method}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Agglomerative clustering with different linkage methods',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. DBSCAN: Density-Based Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) takes a
completely different approach. Instead of distances to centroids, it looks at the
**local density** around each point.

### Key concepts

Given two parameters:
- **eps** ($\varepsilon$) — the radius of the neighbourhood around each point.
- **min_samples** — the minimum number of points within eps to form a dense region.

DBSCAN classifies every point as one of three types:

| Type | Definition |
|------|------------|
| **Core point** | Has at least `min_samples` points within its eps-neighbourhood (including itself) |
| **Border point** | Within eps of a core point, but has fewer than `min_samples` neighbours |
| **Noise point** | Neither core nor border — an outlier |

Clusters are formed by connecting core points that are within eps of each other,
along with their border points.

In [ ]:
# Visualise core, border, and noise points
X_demo, _ = make_blobs(n_samples=200, centers=2, cluster_std=0.8, random_state=42)
# Add some noise points
rng = np.random.RandomState(42)
noise = rng.uniform(low=-6, high=10, size=(20, 2))
X_demo = np.vstack([X_demo, noise])

db = DBSCAN(eps=0.8, min_samples=5)
db_labels = db.fit_predict(X_demo)

# Identify point types
core_mask = np.zeros(len(X_demo), dtype=bool)
core_mask[db.core_sample_indices_] = True
noise_mask = db_labels == -1
border_mask = ~core_mask & ~noise_mask

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X_demo[core_mask, 0], X_demo[core_mask, 1],
           c=db_labels[core_mask], cmap='Set1', s=80, edgecolors='k',
           label='Core points')
ax.scatter(X_demo[border_mask, 0], X_demo[border_mask, 1],
           c=db_labels[border_mask], cmap='Set1', s=80, edgecolors='k',
           marker='s', label='Border points')
ax.scatter(X_demo[noise_mask, 0], X_demo[noise_mask, 1],
           c='grey', s=60, marker='x', linewidths=2,
           label='Noise points')
ax.set_title(f'DBSCAN (eps={db.eps}, min_samples={db.min_samples})')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Core points:   {core_mask.sum()}")
print(f"Border points: {border_mask.sum()}")
print(f"Noise points:  {noise_mask.sum()}")
print(f"Clusters found: {len(set(db_labels) - {-1})}")

---
## 8. DBSCAN Parameters: eps and min_samples

The behaviour of DBSCAN is controlled entirely by `eps` and `min_samples`:

- **Too small eps** — most points are labelled as noise (no dense regions found).
- **Too large eps** — everything merges into one giant cluster.
- **Too small min_samples** — noise gets absorbed into clusters.
- **Too large min_samples** — legitimate points are marked as noise.

A practical rule of thumb: set `min_samples >= dimensionality + 1` (so at least 3
for 2D data) and tune `eps` using a **k-distance plot**.

In [ ]:
# Effect of eps
eps_values = [0.3, 0.5, 0.8, 1.2, 2.0]

fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))

for ax, eps_val in zip(axes, eps_values):
    db_e = DBSCAN(eps=eps_val, min_samples=5)
    labels_e = db_e.fit_predict(X_demo)
    n_clusters = len(set(labels_e) - {-1})
    n_noise = (labels_e == -1).sum()

    unique_labels = set(labels_e)
    colors = [plt.cm.Set1(l / max(len(unique_labels), 1)) if l != -1 else 'grey'
              for l in labels_e]
    ax.scatter(X_demo[:, 0], X_demo[:, 1], c=colors, edgecolors='k', s=30)
    ax.set_title(f'eps={eps_val}\nclusters={n_clusters}, noise={n_noise}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('DBSCAN: effect of eps (min_samples=5)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# k-distance plot to help choose eps
from sklearn.neighbors import NearestNeighbors

k = 5  # min_samples
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_demo)
distances, _ = nn.kneighbors(X_demo)
k_distances = np.sort(distances[:, k - 1])[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_distances, linewidth=2, color='steelblue')
ax.axhline(y=0.8, color='red', linestyle='--', label='Suggested eps ~ 0.8')
ax.set_xlabel('Points (sorted by distance)')
ax.set_ylabel(f'{k}-th nearest neighbour distance')
ax.set_title(f'k-distance plot (k={k}) — look for the "elbow"')
ax.legend()
plt.tight_layout()
plt.show()

The elbow in the k-distance plot suggests that eps around 0.8 is a reasonable
choice. Below that distance, points are densely packed (within clusters); above it,
distances jump sharply (gaps between clusters and noise).

---
## 9. DBSCAN Advantages

DBSCAN has several compelling advantages over K-Means:

1. **No need to specify K** — the number of clusters is determined automatically
   from the data density.
2. **Finds arbitrary shapes** — since it follows density rather than distance to
   centroids, it can discover clusters with any geometry.
3. **Built-in outlier detection** — noise points are flagged with label -1.
4. **Robust to outliers** — unlike K-Means, outliers do not pull centroids away.

Let us see DBSCAN handle shapes that K-Means cannot.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5,
                                     random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# K-Means on moons
km_moons = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moons)
axes[0, 0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons, cmap='Set1',
                    edgecolors='k', s=30)
axes[0, 0].set_title('K-Means on moons')

# DBSCAN on moons
db_moons = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_moons)
axes[0, 1].scatter(X_moons[:, 0], X_moons[:, 1], c=db_moons, cmap='Set1',
                    edgecolors='k', s=30)
axes[0, 1].set_title('DBSCAN on moons')

# K-Means on circles
km_circles = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_circles)
axes[1, 0].scatter(X_circles[:, 0], X_circles[:, 1], c=km_circles, cmap='Set1',
                    edgecolors='k', s=30)
axes[1, 0].set_title('K-Means on circles')

# DBSCAN on circles
db_circles = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_circles)
axes[1, 1].scatter(X_circles[:, 0], X_circles[:, 1], c=db_circles, cmap='Set1',
                    edgecolors='k', s=30)
axes[1, 1].set_title('DBSCAN on circles')

for ax in axes.ravel():
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('K-Means vs DBSCAN on non-spherical data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

DBSCAN perfectly captures the crescent and ring shapes that K-Means misses entirely.

---
## 10. Comprehensive Comparison: K-Means vs Hierarchical vs DBSCAN

Let us compare all three algorithms on four different data shapes: blobs, moons,
circles, and anisotropic blobs.

In [ ]:
# Prepare datasets
n_samples = 300

X_blobs, _ = make_blobs(n_samples=n_samples, centers=3, cluster_std=0.8,
                         random_state=42)
X_moons_c, _ = make_moons(n_samples=n_samples, noise=0.05, random_state=42)
X_circles_c, _ = make_circles(n_samples=n_samples, noise=0.05, factor=0.5,
                               random_state=42)

# Anisotropic
X_aniso, _ = make_blobs(n_samples=n_samples, centers=3, random_state=170)
transformation = np.array([[0.6, -0.6], [-0.4, 0.8]])
X_aniso = X_aniso @ transformation

# Standardise all datasets
datasets = [
    ('Blobs', StandardScaler().fit_transform(X_blobs), 3),
    ('Moons', StandardScaler().fit_transform(X_moons_c), 2),
    ('Circles', StandardScaler().fit_transform(X_circles_c), 2),
    ('Anisotropic', StandardScaler().fit_transform(X_aniso), 3),
]

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 22))

for row, (name, X_ds, n_c) in enumerate(datasets):
    # K-Means
    km_labels = KMeans(n_clusters=n_c, random_state=42, n_init=10).fit_predict(X_ds)
    axes[row, 0].scatter(X_ds[:, 0], X_ds[:, 1], c=km_labels, cmap='Set1',
                          edgecolors='k', s=20)
    axes[row, 0].set_title(f'{name} — K-Means (K={n_c})')

    # Agglomerative
    agg_labels = AgglomerativeClustering(n_clusters=n_c, linkage='ward').fit_predict(X_ds)
    axes[row, 1].scatter(X_ds[:, 0], X_ds[:, 1], c=agg_labels, cmap='Set1',
                          edgecolors='k', s=20)
    axes[row, 1].set_title(f'{name} — Agglomerative (ward)')

    # DBSCAN
    db_labels = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_ds)
    unique_labels = set(db_labels)
    colors_db = [plt.cm.Set1(l / max(len(unique_labels), 1)) if l != -1 else 'grey'
                 for l in db_labels]
    axes[row, 2].scatter(X_ds[:, 0], X_ds[:, 1], c=colors_db,
                          edgecolors='k', s=20)
    n_found = len(unique_labels - {-1})
    axes[row, 2].set_title(f'{name} — DBSCAN (found {n_found} clusters)')

for ax in axes.ravel():
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Clustering algorithm comparison on different data shapes',
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 11. When to Use Which Clustering Method

| Criterion | K-Means | Hierarchical | DBSCAN |
|-----------|---------|-------------|--------|
| **Cluster shape** | Spherical only | Depends on linkage (ward = spherical) | Arbitrary |
| **Must specify K?** | Yes | Yes (or cut dendrogram) | No |
| **Outlier handling** | No (assigns all points) | No (assigns all points) | Yes (labels noise as -1) |
| **Scalability** | Very fast (linear in n) | Slow ($O(n^2)$ memory, $O(n^3)$ time) | Fast for low-dimensional data |
| **Interpretability** | Centroids are easy to interpret | Dendrogram shows hierarchy | Core/border/noise intuitive |
| **Key parameters** | n_clusters | n_clusters, linkage | eps, min_samples |

**Rules of thumb:**
- Start with **K-Means** if clusters are roughly round and you know K.
- Use **hierarchical** if you want to explore different numbers of clusters via the
  dendrogram, or if the data is small enough.
- Use **DBSCAN** if clusters have irregular shapes, outliers are present, or you
  do not know K.

---
## 12. Summary

| Concept | Key point |
|---------|----------|
| Agglomerative clustering | Bottom-up merging into a dendrogram |
| Linkage methods | Ward (compact), single (chaining), complete, average |
| Dendrogram | Tree showing merge history; cut to choose n_clusters |
| DBSCAN | Density-based: core, border, noise points |
| eps and min_samples | Control neighbourhood size and density threshold |
| DBSCAN strengths | Arbitrary shapes, outlier detection, no K needed |
| Algorithm choice | Depends on cluster shape, data size, outlier presence |

---

**Next up:** [03 - Dimensionality Reduction](03_dimensionality_reduction.ipynb)
— PCA and t-SNE for compressing and visualising high-dimensional data.